This notebook gets the English harm direction (h) and refusal direction (r) for a new model, picks which layers to use them at, and saves them in the same file format as the 1.7B ones, so notebooks 05, 06 and 07 can be rerun just by changing `name`. It only reads the saved activation cache, so no model is loaded and it runs fine on the Mac. Directions are always built from English, because English is where the model refuses reliably; other languages are compared against it later.

In [ ]:
from pathlib import Path
import torch
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
name = "qwen3-8b"          # any model with a cache in data/cache/e01/{name}/
ref_lang = "eng_Latn"      # reference language the directions are built from

A = torch.load(ROOT / f"data/cache/e01/{name}/{ref_lang}_acts.pt")
print(A["meta"])
for k in ["harmful_train", "harmless_train", "harmful_val", "harmless_val"]:
    print(f"{k:15s} inst {tuple(A[k]['inst'].shape)}  last {tuple(A[k]['last'].shape)}")   # [N, n_layers, d_model]

n_layers, d_model = A["harmful_train"]["inst"].shape[1:]     # read from data, never hardcoded

The cache has two positions per prompt: `inst` is the last token of the user's instruction (where we read h), and `last` is the last token of the chat template just before the answer starts (where we read r). The direction is a "difference of means": average activation over the harmful train prompts minus average over the harmless ones, at every layer. Whatever the two sets differ in on average is what that vector points to.

In [ ]:
def diff_means(pos):
    """harmful mean - harmless mean at every layer, for one position ('inst' or 'last')."""
    mu_harm = A["harmful_train"][pos].mean(dim=0)     # [N, L, d] -> [L, d]   average over prompts
    mu_safe = A["harmless_train"][pos].mean(dim=0)    # [L, d]
    return mu_harm - mu_safe                          # [L, d]

h = diff_means("inst")     # [n_layers, d_model]   harm direction at t_inst
r = diff_means("last")     # [n_layers, d_model]   refusal direction at the last template token
print(h.shape, r.shape)
print("norm per layer, h:", h.norm(dim=-1)[:: max(1, n_layers // 6)].round().tolist())
print("norm per layer, r:", r.norm(dim=-1)[:: max(1, n_layers // 6)].round().tolist())

Now we check how well each direction separates new prompts it was not built from. We project the val prompts onto the unit direction at each layer (one number per prompt) and measure the gap between harmful and harmless in units of their spread (Cohen's d). Big d means the two sets barely overlap along that direction at that layer. Note that h already separates well at layer 0: early on this is mostly word-level, not understanding.

In [ ]:
def cohens_d(pos, v):
    """Separation of val harmful vs harmless along unit direction v [L, d], per layer -> [L]."""
    v_hat = v / v.norm(dim=-1, keepdim=True)                                 # [L, d] / [L, 1] -> unit rows (broadcasting)
    s1 = torch.einsum("nld,ld->nl", A["harmful_val"][pos], v_hat)           # [N1, L]  dot product per prompt, per layer
    s0 = torch.einsum("nld,ld->nl", A["harmless_val"][pos], v_hat)          # [N0, L]
    return (s1.mean(0) - s0.mean(0)) / ((s1.var(0) + s0.var(0)) / 2).sqrt()  # [L]

d_h = cohens_d("inst", h)
d_r = cohens_d("last", r)

depth = torch.arange(n_layers) / (n_layers - 1)      # 0 = first layer, 1 = last, so models of different size line up
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(range(n_layers), d_h, marker="o", label="h at t_inst")
ax.plot(range(n_layers), d_r, marker="o", label="r at last template token")
ax.set_xlabel("layer"); ax.set_ylabel("Cohen's d on val"); ax.set_title(f"{name}: English val separation"); ax.legend()
plt.show()

Picking layers with a fixed rule so it is the same for every model. For h, the separation curve has a flat top, so we take the middle of the layers that reach 95% of the peak. For r, we take the first layer that reaches 90% of its peak, i.e. where the refusal signal has just formed. On Qwen3-1.7B this rule gives h = 13 and r = 17, versus the 12 and 18 chosen there by actually steering (notebooks 02-03), one layer off each, so it is a good stand-in. The r layer is still provisional until a steering check on the cluster.

In [ ]:
top_h = (d_h >= 0.95 * d_h.max()).nonzero().squeeze(-1)          # layers on the flat top of d_h
h_layer = int(top_h.float().mean().round())                       # middle of that plateau
r_layer = int((d_r >= 0.90 * d_r.max()).nonzero()[0])            # first layer at 90% of peak d_r

print(f"h_layer = {h_layer} (depth {h_layer / (n_layers - 1):.2f}), plateau {top_h.tolist()}")
print(f"r_layer = {r_layer} (depth {r_layer / (n_layers - 1):.2f})")
print(f"d_h at h_layer = {d_h[h_layer]:.2f}, d_r at r_layer = {d_r[r_layer]:.2f}")

Save in the same format as the 1.7B file: `h` is [layers, d], `r` is [positions, layers, d] with only one position here (the last template token), so `dirs["r"][-1, layer]` works exactly as before. The chosen layers go in the same file so other notebooks can read them instead of hardcoding. If a file already exists (for example the 1.7B one from E0) it is not overwritten; we only compare.

In [ ]:
out = ROOT / f"data/cache/e00/{name}_directions.pt"
dirs = {"h": h, "r": r[None],                         # r: [L, d] -> [1, L, d]   (position axis, like E0)
        "h_layer": h_layer, "r_layer": r_layer,
        "note": f"diff of means on e01 {ref_lang} train; h at t_inst, r at last template token; layers by rule in nb08"}

if out.exists():
    old = torch.load(out)
    cos = lambda a, b: (a * b).sum(-1) / (a.norm(dim=-1) * b.norm(dim=-1))          # [L]
    print("exists, not overwritten. min cos with saved: h", cos(h, old["h"]).min().item(),
          " r", cos(r, old["r"][-1]).min().item())
else:
    torch.save(dirs, out)
    print("saved", out.relative_to(ROOT), {k: tuple(v.shape) for k, v in dirs.items() if torch.is_tensor(v)})

Quick check that the saved file loads and the unit directions really are unit length, then a look at how h and r relate: their cosine per layer. In 1.7B they were related but not the same (h is "harmful content", r is "I will refuse"), and we expect the same here.

In [ ]:
dirs = torch.load(out)
h_hat = dirs["h"] / dirs["h"].norm(dim=-1, keepdim=True)          # [L, d]
r_hat = dirs["r"][-1] / dirs["r"][-1].norm(dim=-1, keepdim=True)  # [L, d]
print("unit norms ok:", torch.allclose(h_hat.norm(dim=-1), torch.ones(n_layers)))
cos_hr = (h_hat * r_hat).sum(-1)                                  # [L]  cosine per layer
print("cos(h, r) per layer:", cos_hr[:: max(1, n_layers // 6)].round(decimals=2).tolist())
print("layers in file:", dirs.get("h_layer"), dirs.get("r_layer"))